# SwiftLog Week 5: Machine Learning Engineering

**Mission:** Intelligent Logistics Prediction & Supply Chain Optimization.

This notebook reviews source data, engineered features, model comparisons, evaluation artefacts, and limitations. The full model-training pipeline is `../run_swiftlog_project.py`. Run that script from the project root to regenerate the saved models and outputs before rerunning this notebook.

The supplied data is used as provided for this educational prototype. Results are reported as measured; weak performance is not hidden or described as production-ready.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
DATA = ROOT / 'Data'
PROCESSED = DATA / 'Processed_Datasets'
MODELS = ROOT / 'Models'


## 2. Source data and quality checks
The source tables cover customers, orders, order items, products, shipments, delivery performance, returns, inventory, warehouses, suppliers, and employees. The supplied file-level data dictionary is brief; calculated model features are documented in `Feature_Data_Dictionary.csv`.

In [ ]:
quality_path = PROCESSED / 'data_quality_summary.csv'
if quality_path.exists():
    quality = pd.read_csv(quality_path)
    display(quality)
else:
    print('Data-quality summary is not available.')


## 3. Inspect the original target distributions

In [ ]:
original = DATA / 'Original_Datasets'
shipments = pd.read_csv(original / 'SwiftLog_Shipments.csv')
returns = pd.read_csv(original / 'SwiftLog_Returns.csv')
orders = pd.read_csv(original / 'SwiftLog_Orders.csv')

print('Shipments:', len(shipments))
if 'DelayDays' in shipments.columns:
    delays = pd.to_numeric(shipments['DelayDays'], errors='coerce')
    print('Delayed shipments (DelayDays > 0):', int((delays > 0).sum()))
if 'OrderID' in returns.columns:
    print('Orders associated with recorded returns:', returns['OrderID'].nunique())
if 'OrderID' in orders.columns:
    print('Orders:', orders['OrderID'].nunique())


## 4. Engineered feature samples

The delivery-delay target is `DelayDays > 0`. The return target is whether an order appears in the Returns table. Demand features use weekly product quantities and lagged values from prior weeks. Customer value is framed as next-month revenue, using the preceding three months of activity. For the demand and customer-value tasks, evaluation is chronological rather than a random row split.

### Delivery delay features

In [ ]:
feature_path = PROCESSED / 'delivery_delay_features.csv'
if feature_path.exists():
    features = pd.read_csv(feature_path)
    print('Shape:', features.shape)
    display(features.head())
else:
    print('Feature file not found:', feature_path)


### Product return features

In [ ]:
feature_path = PROCESSED / 'product_return_features.csv'
if feature_path.exists():
    features = pd.read_csv(feature_path)
    print('Shape:', features.shape)
    display(features.head())
else:
    print('Feature file not found:', feature_path)


### Weekly demand forecasting features

In [ ]:
feature_path = PROCESSED / 'demand_forecasting_features.csv'
if feature_path.exists():
    features = pd.read_csv(feature_path)
    print('Shape:', features.shape)
    display(features.head())
else:
    print('Feature file not found:', feature_path)


### Customer next-month value features

In [ ]:
feature_path = PROCESSED / 'customer_value_features.csv'
if feature_path.exists():
    features = pd.read_csv(feature_path)
    print('Shape:', features.shape)
    display(features.head())
else:
    print('Feature file not found:', feature_path)


## 5. Compare the candidate algorithms

The table contains candidate algorithms per task. `Selected` marks the model stored in `Best_Model.pkl`; `Tuned` indicates that a tuning search was accepted, where recorded. The selected model metrics are stored in the comparison table.

In [ ]:
comparison = pd.read_excel(MODELS / 'Model_Comparison.xlsx')
display(comparison)


## 6. Review selected models and held-out metrics

In [ ]:
if 'Selected' in comparison.columns:
    selected = comparison[comparison['Selected'] == True]
    display(selected)
else:
    print('The comparison table does not contain a Selected column.')


## 7. Evaluation visuals

In [ ]:
from IPython.display import Image, display

for name in [
    'delivery_delay_classification_confusion_matrix.png',
    'product_return_classification_confusion_matrix.png',
]:
    path = MODELS / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

importance = MODELS / 'Feature_Importance.png'
if importance.exists():
    display(Image(filename=str(importance)))


## 8. Interpretation and limitations

- Delivery-delay and return-classification ROC-AUC values near 0.5 indicate weak class discrimination, even when F1 or accuracy appears high.
- Negative held-out R-squared for demand or next-month customer value means the selected model performed worse than a simple mean-prediction baseline on the test period.
- The dataset spans a limited period and has limited operational context. Forecast quality may improve with richer timestamps, planned delivery dates, inventory snapshots, and more history.
- Model outputs should support human review only. They are not validated for live logistics decisions.

## 9. Reproduce the outputs

From the project root, run `python run_swiftlog_project.py`. The script preprocesses the source data, builds the features, compares candidate algorithms, evaluates candidates, saves model bundles and visualizations, and regenerates reports and the presentation. Then rerun this notebook to inspect the refreshed outputs.